# Week 14: Agent testing and human oversight

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Purpose:** a fully worked, self-paced in-class practice with an optional extension. Weeks 11 and 18 are optional examination-readiness materials. These notebooks do not create weekly graded assignments.

**Before running:** choose a Python 3 kernel in Jupyter or a CPU runtime in Colab. Run every cell from top to bottom with Shift+Enter. Setup installs missing free packages. No API key or external dataset download is required.

**Learning target:** complete the calculation, inspect intermediate values, and explain why the protocol supports a limited claim.

**Time:** about 25-45 minutes including reading, prediction, execution, and explain-back.


## 1. Set up and record the environment

Keep the seed fixed for the first run. Record actual outputs and package versions rather than copying a claimed performance number.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** The environment cell must finish before any analysis cell. Restart and run all if a variable is missing.


## 2. Declare permissions and a finite budget

Use an offline state machine. It demonstrates agent safety boundaries without requiring paid inference or accessing real files.


In [ ]:
policy={"allowed_tools":{"add"},"budget":3}
def execute_plan(plan):
    trace=[]
    if len(plan)>policy["budget"]: return {"status":"budget exceeded","trace":trace}
    for request in plan:
        if request.get("tool") not in policy["allowed_tools"]:
            return {"status":"needs human review","trace":trace}
        values=request.get("values")
        if not isinstance(values,list) or not values or not all(type(v) in (int,float) and np.isfinite(v) for v in values):
            return {"status":"invalid input","trace":trace}
        trace.append({"tool":"add","result":float(sum(values))})
    return {"status":"complete","trace":trace}


**Checkpoint.** No tool can be enabled merely by a sentence in retrieved text.


## 3. Probe boundary conditions

Test valid actions, forbidden actions, invalid data, and an over-budget plan as separate failure modes.


In [ ]:
tests=[([{"tool":"add","values":[1,2]}],"complete"),
       ([{"tool":"send_email","values":[1]}],"needs human review"),
       ([{"tool":"add","values":["ignore rules"]}],"invalid input"),
       ([{"tool":"add","values":[1]}]*4,"budget exceeded")]
results=[execute_plan(plan)["status"] for plan,_ in tests]
assert results==[expected for _,expected in tests]
print(results)


**Checkpoint.** The tests establish behavior only for this deterministic implementation and the supplied cases.


## 4. Write an oversight record

Capture what a human must inspect before granting a new capability. These are simulated operations, so no external message is sent.


In [ ]:
summary={"tests_passed":len(tests),"simulated":True,"allowed_tools":sorted(policy["allowed_tools"]),
         "human_review":"Inspect tool identity, arguments, purpose, and consequence before changing permissions."}
print(json.dumps(summary,indent=2))


**Checkpoint.** Explain how audit logs, stopping rules, and user approval differ from a vague instruction to be careful.


## 6. Preserve your evidence

Export a JSON record with a seed, package version, and measured result. Download the artifacts directory before leaving Colab; runtime storage is temporary.


In [ ]:
record={"week":14,"seed":SEED,"numpy":np.__version__,"python":platform.python_version(),"result":summary}
(ARTIFACTS/"week-14-evidence.json").write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


**Checkpoint.** The evidence file must match the result you actually obtained. A saved screenshot alone does not permit rerunning the calculation.


## 7. Try one change and explain it
1. Predict the effect of one change before editing code. Change one parameter, input, or boundary condition only.
2. Rerun all cells in a fresh kernel. Keep the original evidence record and compare the observed difference.
3. Explain the result using the mechanism, not just the final number.
4. Name one assumption that limits the claim, and a new test that could challenge it.

**Troubleshooting:** if installation fails, restore network access; if a variable is undefined, run earlier cells; if your output differs, compare package versions, seed, and data construction. Do not repeatedly edit code until a preferred score appears.

**Completion criterion:** one executed notebook, one evidence JSON, and a brief claim/evidence/limitation statement.

### Reference documentation
- [NumPy user guide](https://numpy.org/doc/stable/user/)
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html)
- [Course repository](https://github.com/CHjonnyYu/MLES-Fall-2026)
